In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH_VAL_CLIM = 'Valores_Climatologicos_1970_2024_con_Coordenadas.csv'

In [3]:
COLS_NOMBRES = [
    'indicativo',
    'nombre',
    'provincia',
]

COLS_INTS = [
    'altitud'
]

COLS_FLOATS = [
    'tmed',
    'prec',
    'tmin',
    'tmax',
    'velmedia',
    'sol',
    'presMax',
    'presMin',
    'hrMedia',
    'dir',
    'racha',
    'hrMax',
    'hrMin'
]

COLS_NUMS = COLS_INTS + COLS_FLOATS

COL_FECHA = 'fecha'
COLS_HORAS = [
    'horaPresMax',
    'horaPresMin',
    'horatmin',
    'horatmax',
    'horaracha',
    'horaHrMax',
    'horaHrMin'
]

COLS_COORDS = [
    'lat',
    'lon'
]

In [4]:
df = pd.read_csv(PATH_VAL_CLIM,
                 sep=';', decimal = ',',
                 dtype = {nombre: 'string' for nombre in COLS_NOMBRES} |
                         {entero: 'Int64' for entero in COLS_INTS} |
                         {coordenada: 'Float64' for coordenada in COLS_COORDS} |
                         {otros: 'object' for otros in COLS_FLOATS + COLS_HORAS},
                         # {decimal: 'Float64' for decimal in COLS_FLOATS if decimal != 'prec'},
                 parse_dates = [COL_FECHA])

In [5]:
display(df)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,horatmax,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat
0,1970-01-01,C249I,FUERTEVENTURA AEROPUERTO,LAS PALMAS,25,"19,8","0,0","16,0","23,5","6,7",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-13.863056,28.444722
1,1970-01-01,1679A,MONFORTE DE LEMOS,LUGO,291,"4,0","0,0","0,0","8,0",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-7.510833,42.531667
2,1970-01-01,2462,PUERTO DE NAVACERRADA,MADRID,1893,"-5,0","0,4","-7,0","-3,0","3,1",...,15:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.010556,40.793056
3,1970-01-01,1212E,ASTURIAS AEROPUERTO,ASTURIAS,127,"4,3","2,6","2,0","6,6","0,0",...,01:00,99.0,"7,2",00:13,NaN,NaN,NaN,NaN,-6.044167,43.566944
4,1970-01-01,0016A,REUS AEROPUERTO,TARRAGONA,71,"5,5","0,0","0,6","10,4","1,7",...,14:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.163611,41.145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7427566,2024-12-31,9562X,MORELLA,CASTELLON,990,"4,8","0,0","0,8","8,8","1,4",...,13:42,12.0,"5,0",16:50,97.0,23:50,63.0,14:00,-0.101944,40.621667
7427567,2024-12-31,1002Y,"BAZTAN, IRURITA",NAVARRA,183,"4,2","0,0","-4,9","13,4","0,0",...,14:50,22.0,"3,1",16:20,100.0,09:00,50.0,14:10,-1.543056,43.135833
7427568,2024-12-31,3254Y,MORA,TOLEDO,717,"2,0","0,0","-3,1","7,1","1,1",...,11:18,20.0,"3,9",14:30,100.0,09:20,80.0,11:20,-3.780556,39.686944
7427569,2024-12-31,0194D,"CORBERA, PUIG D'AGULLES",BARCELONA,647,NaN,"0,0",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.885,41.408056


### Preprocesado del Dataframe

In [6]:
df_prep = df.copy(deep = True)

In [7]:
COL_PREC = 'prec'
# df_prep.loc[df[COL_PREC] == 'Ip', COL_PREC] = 0
df_prep = df_prep[df_prep[COL_PREC] != 'Acum']
df_prep[COL_PREC] = df_prep[COL_PREC].replace('Ip', 0)

In [8]:
# Reemplazamos las comas por puntos para poder cargar los números como decimales (floats)
df_prep[COLS_FLOATS] = df_prep[COLS_FLOATS].replace(',', '.', regex=True).apply(pd.to_numeric, errors = 'raise')

In [9]:
df_prep.dtypes

fecha          datetime64[ns]
indicativo     string[python]
nombre         string[python]
provincia      string[python]
altitud                 Int64
tmed                  float64
prec                  float64
tmin                  float64
tmax                  float64
velmedia              float64
sol                   float64
presMax               float64
horaPresMax            object
presMin               float64
horaPresMin            object
hrMedia               float64
horatmin               object
horatmax               object
dir                   float64
racha                 float64
horaracha              object
hrMax                 float64
horaHrMax              object
hrMin                 float64
horaHrMin              object
lon                   Float64
lat                   Float64
dtype: object

In [10]:
# Corregimos inconsistencias en nombres de provincias
df_prep.loc[df_prep['provincia'] == 'BALEARES', 'provincia'] = 'ILLES BALEARS'
df_prep.loc[df_prep['provincia'] == 'STA. CRUZ DE TENERIFE', 'provincia'] = 'SANTA CRUZ DE TENERIFE'
# df_prep.loc[df_prep['provincia'] == 'SANTA CRUZ DE TENERIFE', 'provincia'] = 'STA. CRUZ DE TENERIFE'

In [11]:
# Borramos dos datos de una medición anómala
df_prep.loc[
    (df_prep['indicativo'] == '6084X') &
    (df_prep['tmin'] == 50.0) &
    (df_prep['tmax'] == -50.0),
    ['tmax', 'tmin']
] = np.nan

In [12]:
# Procesamos dirección del viento (según la info proporcionada en el fichero de metadatos)
dict_dir = {
    88: 'Desconocida',
    99: 'Varias'
}
dir_por_defecto = 'Válida'

df_prep['dir_tipo'] = df_prep['dir'].map(dict_dir).fillna(dir_por_defecto)

In [13]:
# Añadimos el sufijo :00 a las columnas cuyos valores carecen de él
cols_sufijo = ['horaPresMax', 'horaPresMin']
df_prep[cols_sufijo] = df_prep[cols_sufijo].where(
    df_prep[cols_sufijo].isna() | (df_prep[cols_sufijo] == 'Varias'),
    df_prep[cols_sufijo] + ':00'
)

In [14]:
df_prep.sample(20)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,tmax,velmedia,...,dir,racha,horaracha,hrMax,horaHrMax,hrMin,horaHrMin,lon,lat,dir_tipo
4800580,2016-08-31,1701X,RIBADAVIA,OURENSE,112,23.2,0.0,13.9,32.6,NaN,...,NaN,NaN,NaN,94.0,06:10,35.0,16:10,-8.129167,42.3,Válida
5481031,2018-11-12,4489X,ALCONCHEL,BADAJOZ,170,12.9,0.0,10.4,15.4,1.4,...,99.0,5.0,00:30,99.0,Varias,76.0,15:00,-7.281944,38.484722,Varias
4517859,2015-09-27,5796,MORÓN DE LA FRONTERA,SEVILLA,87,23.9,0.0,14.4,33.4,1.4,...,99.0,5.8,Varias,60.0,06:20,21.0,Varias,-5.611389,37.164444,Varias
557666,1982-09-09,3100B,ARANJUEZ,MADRID,540,24.0,0.0,15.0,33.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.546111,40.067222,Válida
3457385,2012-02-10,4464X,ALBURQUERQUE,BADAJOZ,283,5.7,0.0,-4.1,15.5,1.1,...,36.0,7.8,14:30,94.0,Varias,23.0,15:00,-6.995278,39.181944,Válida
5595777,2019-03-26,1010X,BERA,NAVARRA,54,7.9,0.0,1.4,14.4,1.7,...,35.0,7.5,13:20,94.0,Varias,40.0,11:40,-1.675833,43.278611,Válida
7161633,2024-03-06,9772X,LA POBLA DE CÉRVOLES,LLEIDA,673,8.7,0.0,2.1,15.3,NaN,...,NaN,NaN,NaN,75.0,07:10,39.0,Varias,0.913611,41.366111,Válida
2121111,2005-01-22,4527X,AROCHE,HUELVA,267,10.0,0.0,-0.2,20.1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.975833,37.980278,Válida
3040122,2010-07-21,7066Y,"YESTE, EMBALSE FUENSANTA",ALBACETE,680,27.0,0.0,18.3,35.7,NaN,...,NaN,NaN,NaN,81.0,06:50,20.0,15:30,-2.218889,38.393333,Válida
5862061,2020-01-31,1696O,BEARIZ,OURENSE,610,12.5,30.2,10.6,14.4,NaN,...,NaN,NaN,NaN,100.0,Varias,89.0,14:50,-8.278056,42.468056,Válida


In [15]:
df_prep.to_csv('Valores_Climatologicos_1970_2024_Limpios.csv', sep = ';', index = False)